# Imports and Agents Initialization

In [ ]:
import networkx as nx
import json

In [ ]:
# setup ollama to run llm locally
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# run ollama in background to pull a model
def run_ollama_in_background():
  import threading
  import subprocess
  import time

  def ollama_serve():
      subprocess.Popen(["ollama", "serve"])

  thread = threading.Thread(target=ollama_serve)
  thread.start()
  time.sleep(5)

run_ollama_in_background()

In [ ]:
# pull models
!ollama pull gemma3
# !ollama pull llava
# !ollama pull phi3

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠏ pulling manifest ⠏ pulling manifest 
pulling aeda25e63ebd:   0% ▕▏ 4.9 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   1% ▕▏  22 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   2% ▕▏  62 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   3% ▕▏ 101 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   4% ▕▏ 133 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   5% ▕▏ 152 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   6% ▕▏ 183 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   6% ▕▏ 210 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   7% ▕▏ 223 MB/3.3 GB                  pulling manifest 
pulling aeda25e63ebd:   8% ▕▏ 251 MB/3.3 GB                  pulling manifest 
pu

In [ ]:
# install python dependencies
!pip install ollama
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 5.8 MB/s eta 0:00:00


In [ ]:
# --- LLM Model Functions ---

def groq_deepseek_r1(messages):
    from groq import Groq
    import re
    client = Groq(api_key="API-Key")
    response = client.chat.completions.create(
        model="deepseek-r1-distill-llama-70b",
        messages=messages,
        temperature=0,
    )
    think_pattern = r"<think>(.*?)</think>"
    content_pattern = r"</think>\s*(.*)"

    # Extracting the 'thinking' content using regex
    think_match = re.search(think_pattern, response.choices[0].message.content.strip(), re.DOTALL)
    content_match = re.search(content_pattern, response.choices[0].message.content.strip(), re.DOTALL)

    # Assign variables based on regex match results
    thinking_content = think_match.group(1).strip() if think_match else None
    main_content = content_match.group(1).strip() if content_match else None
    return main_content, thinking_content

def groq_qwen_qwq(messages):
    from groq import Groq
    import re
    client = Groq(api_key="API-Key")
    response = client.chat.completions.create(
        model="qwen-qwq-32b",
        messages=messages,
        temperature=0,
    )
    think_pattern = r"<think>(.*?)</think>"
    content_pattern = r"</think>\s*(.*)"

    # Extracting the 'thinking' content using regex
    think_match = re.search(think_pattern, response.choices[0].message.content.strip(), re.DOTALL)
    content_match = re.search(content_pattern, response.choices[0].message.content.strip(), re.DOTALL)

    # Assign variables based on regex match results
    thinking_content = think_match.group(1).strip() if think_match else None
    main_content = content_match.group(1).strip() if content_match else None
    return main_content, thinking_content

def groq_llama_4scout(messages):
    from groq import Groq
    client = Groq(api_key="API-Key")
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content.strip(), None

def groq_mistral_saba(messages):
    from groq import Groq
    client = Groq(api_key="API-Key")
    response = client.chat.completions.create(
        model="mistral-saba-24b",
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content.strip(), None

def groq_llama_33versatile(messages):
    from groq import Groq
    client = Groq(api_key="API-Key")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content.strip(), None

def openrouter_reka(messages):
    from openai import OpenAI
    import re
    client = OpenAI(
        api_key="API-Key",
        base_url="https://openrouter.ai/api/v1"
    )
    response = client.chat.completions.create(
        model="rekaai/reka-flash-3:free",
        messages=messages,
        temperature=0
    )
    think_pattern = r"<reasoning>(.*?)</reasoning>"
    content_pattern = r"</reasoning>\s*(.*)"

    # Extracting the 'thinking' content using regex
    think_match = re.search(think_pattern, response.choices[0].message.content.strip(), re.DOTALL)
    content_match = re.search(content_pattern, response.choices[0].message.content.strip(), re.DOTALL)

    # Assign variables based on regex match results
    thinking_content = think_match.group(1).strip() if think_match else None
    main_content = content_match.group(1).strip() if content_match else None
    return main_content, thinking_content

def openrouter_maids(messages):
    from openai import OpenAI
    client = OpenAI(
        api_key="API-Key",
        base_url="https://openrouter.ai/api/v1"
    )
    response = client.chat.completions.create(
        model="microsoft/mai-ds-r1:free",
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content.strip(), None

def ollama_mistral(messages):
    from ollama import chat
    response = chat("mistral", messages=messages)
    return response["message"]["content"].strip(), None

def ollama_gemma3(messages):
    from ollama import chat
    response = chat("gemma3", messages=messages)
    return response["message"]["content"].strip(), None

def ollama_llava(messages):
    from ollama import chat
    response = chat("llava", messages=messages)
    return response["message"]["content"].strip(), None

def ollama_phi3(messages):
    from ollama import chat
    response = chat("phi3", messages=messages)
    return response["message"]["content"].strip(), None

def perplexity_sonar(messages):
    import requests
    url = "https://api.perplexity.ai/chat/completions"
    payload = {
        "model": "sonar",
        "messages": messages
    }
    headers = {
        "Authorization": "your-data",
        "Content-Type": "application/json"
    }
    response = requests.request("POST", url, json=payload, headers=headers).json()
    return response["choices"][0]["message"]["content"].strip()

def google_gemini_25flash(messages):
    from openai import OpenAI
    client = OpenAI(
        api_key="API-Key",
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
    response = client.chat.completions.create(
        model="gemini-2.5-flash-preview-04-17",
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content.strip(), None

def x_grok_3mini(messages):
    from openai import OpenAI
    client = OpenAI(
        api_key="API-Key",
        base_url="https://api.x.ai/v1"
    )
    response = client.chat.completions.create(
        model="grok-3-mini-beta",
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content.strip(), response.choices[0].message.reasoning_content

def openai_gpt_35turbo(messages):
    from openai import OpenAI
    client = OpenAI(api_key="API-Key")
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content.strip(), None

def openai_gpt4(messages):
    from openai import OpenAI
    client = OpenAI(api_key="API-Key")
    response = client.chat.completions.create(
        model="gpt-4",
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content.strip(), None

models = {
    "groq_deepseek_r1": groq_deepseek_r1,
    "groq_qwen_qwq": groq_qwen_qwq,
    "groq_llama_4scout": groq_llama_4scout,
    "groq_llama_33versatile": groq_llama_33versatile,
    "groq_mistral_saba": groq_mistral_saba,
    "ollama_gemma3": ollama_gemma3,
    "google_gemini_25flash": google_gemini_25flash,
    "x_grok_3mini": x_grok_3mini,
    "openai_gpt4": openai_gpt4,
    "openai_gpt_35turbo": openai_gpt_35turbo,
}

# Utility Codes

In [ ]:
# --- Request an opinion on the following topic ---
def get_opinion(topic, model_fn):
    messages = [
        {
            "role": "system",
            "content": "You shuold provide your personal opinion, along with a brief explanation of why you hold that view. Your response should be clear and concise."
        },
        {"role": "user", "content": topic},
        {"role": "user", "content": "What’s your opinion on this statement? Share your thoughts in one sentence."}
    ]
    try:
        return model_fn(messages)
    except Exception as e:
        return f"error: {str(e)}", None


# --- Request a vote on the following topic ---
def get_vote(topic, opinions, model_fn):
    messages = [
        {
            "role": "system",
            "content": "Form your own view on the topic after considering the peer's opinion, and then respond with 'agree' or 'disagree' (one word, no extra text) to the topic"
        },
        {"role": "user", "content": topic},
        {"role": "user", "content": opinions}
    ]
    try:
        return model_fn(messages)
    except Exception as e:
        return f"error: {str(e)}", None

In [ ]:
# --- Generate a Regular Graph using NetworkX ---
def generate_networkx_regular_graph(degree=3, num_nodes=10):
    if degree >= num_nodes or (degree * num_nodes) % 2 != 0:
        return f"Invalid degree {degree} for {num_nodes} nodes."

    try:
        G = nx.random_regular_graph(degree, num_nodes)
        return {node: list(G.adj[node]) for node in G.nodes()}
    except nx.NetworkXError as e:
        return f"NetworkX error: {e}"

In [ ]:
# --- Simulation process ---
def collect_opinions(topic, num_agents=10):
    print(f"\n--- {topic} ---")
    print(f"\n--- Opinions phase ---")
    print(f"Models: {models.keys()}")

    opinions = []

    for model_name in models.keys():
        for i in range(num_agents):
            opinion, thinking = get_opinion(topic, models[model_name])
            print(f"\n----- {model_name}_{i} -----")
            print(f"# Opinion: {opinion}")
            print(f"# Thinking: {thinking}")
            print(f"----- ---------------- -----\n")
            opinions.append({"agent": f"{model_name}_{i}", "opinion": opinion, "thinking": thinking})
            time.sleep(1)

    return opinions


def collect_votes(topic, opinions, network_degree=3, num_agents=10):
    print(f"\n--- {topic} ---")
    print(f"\n--- Voting phase ---")
    print(f"Models: {models.keys()}")
    print(f"Network Degree: {network_degree}")
    print(f"Opinion Network:")

    opinions_network = generate_networkx_regular_graph(degree=network_degree, num_nodes=num_agents*len(models))
    print(opinions_network)

    network_voting_result = {'network_degree': network_degree, 'opinions_network': opinions_network, 'voting_result': []}
    agent_count = 0

    for model_name in models.keys():
        agent_votes = []

        for i in range(num_agents):

            opinion_indexes = opinions_network[agent_count]
            opinions_text = ""
            for opinion_index in opinion_indexes:
                opinions_text += f"{opinions[opinion_index]['agent']}: {opinions[opinion_index]['opinion']}\n"

            vote, thinking = get_vote(topic, opinions_text, models[model_name])
            print(f"\n----- {model_name}_{i} -----")
            print(f"# Vote: {vote}")
            print(f"# Opinions:\n{opinions_text.strip()}")
            print(f"# Thinking: {thinking}")
            print(f"----- ---------------- -----\n")

            agent_votes.append({"agent": f"{model_name}_{i}", "vote": vote, "opinions": opinions_text.strip(), "thinking": thinking})
            agent_count += 1
            time.sleep(1)

        network_voting_result["voting_result"].append({"model": model_name, "votes": agent_votes})

    # print(f"Vote Summary: {network_voting_result}")
    return network_voting_result

# Main Code

In [ ]:
# --- Opinion Phase ---
import json

# run ollama in background to run a model
run_ollama_in_background()

topic = "Protecting the environment is the individual's responsibility."

if __name__ == "__main__":
    opinions = collect_opinions(topic)
    with open('opinions.json', 'w', encoding="utf-8") as f:
      json.dump(opinions, f, ensure_ascii=False, indent=4)


--- Protecting the environment is the individual's responsibility. ---

--- Opinions phase ---
Models: dict_keys(['groq_deepseek_r1', 'groq_qwen_qwq', 'groq_llama_4scout', 'groq_llama_33versatile', 'groq_mistral_saba', 'ollama_gemma3', 'google_gemini_25flash', 'x_grok_3mini', 'openai_gpt4', 'openai_gpt_35turbo'])

----- groq_deepseek_r1_0 -----
# Opinion: I agree that individuals have a responsibility to protect the environment, as their collective actions can significantly contribute to positive change, though systemic and institutional efforts are also essential for comprehensive impact.
# Thinking: Okay, so I need to figure out my opinion on whether protecting the environment is the individual's responsibility. Hmm, where do I start? Well, I know that the environment is a big deal these days with all the talk about climate change and pollution. But is it really up to each person to make a difference?

I guess individuals can do things like recycle, use public transport, or buy eco-

In [ ]:
# --- Voting Phase ---
import json

# run ollama in background to run a model
run_ollama_in_background()

with open('opinions.json', "r", encoding="utf-8") as f:
    opinions = json.load(f)

topic = "Protecting the environment is the individual's responsibility."
result = {"topic": topic, "opinions": opinions, "result": []}

if __name__ == "__main__":
  for degree in range(1, 91):
      result["result"].append(collect_votes(topic, opinions, network_degree=degree))
      with open('network_voting_result.json', 'w', encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=4)


Выходные данные были обрезаны до нескольких последних строк (5000).
groq_llama_33versatile_2: I strongly agree with this statement because I believe that every individual's small actions, such as reducing waste and conserving resources, can collectively make a significant impact on protecting the environment.
# Thinking: First, the system prompt instructs me to form my own view on the topic after considering the peer's opinion, and then respond with just 'agree' or 'disagree' (one word, no extra text) to the topic.

The topic is: "Protecting the environment is the individual's responsibility."

Now, I need to consider the peers' opinions. The human provided a list of various responses from different AI models. Most of them agree, but with nuances:

- Many strongly agree, emphasizing that individual actions can collectively make a big impact.

- Some agree but stress that it's shared with governments and corporations.

- A few disagree or partially disagree, like groq_llama_4scout_0 who

KeyboardInterrupt: 